# Gradient obfuscation

This code trains a MobileNetV2 model on CIFAR-10 with gradient obfuscation and evaluates its performance on both clean and adversarial images.

This setup prepares a CIFAR-10 experiment to support a gradient-obfuscation demo. After importing TensorFlow/Keras, the CIFAR-10 dataset (60,000 colour images across 10 classes) is loaded. A preprocessing function resizes each image to 64×64, scales pixel values to [0, 1], and converts labels to one-hot vectors (squeezed to drop the extra dimension). The data is fed through efficient tf.data pipelines: mapping the preprocessing, shuffling the training set, batching at 32, and using prefetch(tf.data.AUTOTUNE) to overlap input preparation with model execution. The model constructor builds a MobileNetV2 with weights=None (training from scratch), include_top=True, and classes=10 for direct classification on the resized inputs, wrapped in a simple Sequential container. This combination gives a fast, modern CNN baseline with clean input pipelines—useful for illustrating how certain “defences” can affect gradients in subsequent sections.

In [1]:
import tensorflow as tf
from tensorflow.keras import models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.datasets import cifar10
import numpy as np

# Load CIFAR-10 data
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Preprocessing function
def preprocess_image(image, label):
    image = tf.image.resize(image, (64, 64))
    image = tf.cast(image, tf.float32) / 255.0
    label = tf.squeeze(tf.one_hot(label, depth=10))
    return image, label

# tf.data pipelines
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.map(preprocess_image).shuffle(10000).batch(32).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_ds = test_ds.map(preprocess_image).batch(32).prefetch(tf.data.AUTOTUNE)

# MobileNetV2 model
def create_mobilenetv2():
    base_model = MobileNetV2(
        weights=None,
        input_shape=(64, 64, 3),
        include_top=True,
        classes=10
    )
    model = models.Sequential([base_model])
    return model

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 598s 4us/step


2026-10-26 22:07:58.237753: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


The PGD attack function generates adversarial examples via projected gradient descent. Wrapped with @tf.function for graph execution, it iteratively (for iters steps) computes the gradient of the categorical cross-entropy loss with respect to the current images, takes a step in the sign of that gradient scaled by a step size alpha, and then projects the result back into the ε-ball around the original images (via tf.clip_by_value(images−ε, images+ε)) as well as the valid pixel range [0,1]. The parameters govern attack strength: epsilon bounds total perturbation, alpha is the per-iteration step, and iters is the number of attack steps. This yields perturbed inputs that remain visually similar yet are optimised to increase the model’s loss.
The custom training loop with gradient obfuscation fits the model using Adam and categorical cross-entropy, but deliberately clips every gradient to the small range [-1e−2, 1e−2] before applying updates. This “defence” suppresses gradient magnitude and can make gradient-based attacks appear to fail (by providing weak or noisy gradients) without conferring true robustness—classic gradient obfuscation. The loop reports loss every 100 steps so you can monitor optimisation progress while noting that any apparent robustness must be validated with stronger, gradient-free or expectation-over-transformation attacks to rule out obfuscation.

In [2]:
# PGD attack implementation
@tf.function
def pgd_attack(model, images, labels, epsilon=0.03, alpha=0.007, iters=5):
    adv_images = images
    for _ in tf.range(iters):
        with tf.GradientTape() as tape:
            tape.watch(adv_images)
            predictions = model(adv_images)
            loss = tf.keras.losses.categorical_crossentropy(labels, predictions)
        gradients = tape.gradient(loss, adv_images)
        adv_images = adv_images + alpha * tf.sign(gradients)
        adv_images = tf.clip_by_value(adv_images, images - epsilon, images + epsilon)
        adv_images = tf.clip_by_value(adv_images, 0.0, 1.0)
    return adv_images

# Custom training loop with gradient obfuscation
def train_with_gradient_obfuscation(model, dataset, epochs=2):
    optimizer = tf.keras.optimizers.Adam()
    loss_fn = tf.keras.losses.CategoricalCrossentropy()

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        for step, (x_batch, y_batch) in enumerate(dataset):
            with tf.GradientTape() as tape:
                predictions = model(x_batch, training=True)
                loss = loss_fn(y_batch, predictions)

            gradients = tape.gradient(loss, model.trainable_variables)
            # Apply gradient obfuscation
            gradients = [tf.clip_by_value(g, -1e-2, 1e-2) for g in gradients]
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))

            if step % 100 == 0:
                print(f"Step {step}, Loss: {loss.numpy():.4f}")

The evaluation function tests model performance on either clean or adversarial inputs. It takes a single batch from the dataset (dataset.take(1)), optionally crafts adversarial examples for that batch using the PGD attack, and then calls model.evaluate to obtain loss and accuracy. The result is printed as a percentage and returned. Note that sampling just one batch makes this a quick demo rather than a full test-set evaluation; it’s useful for illustrating the effect of adversarial perturbations but not for reporting definitive metrics.
The demo runner builds and compiles a MobileNetV2 classifier, trains it for two epochs using the gradient-obfuscation loop (which clips gradients to a tiny range), and then evaluates accuracy on both clean and adversarial batches. The contrast between the two printed accuracies helps reveal whether any apparent robustness stems from genuine stability or from gradient obfuscation—i.e., gradients made uninformative for attack optimisation rather than the model actually learning robust features.

In [3]:
# Evaluation function
def evaluate_model(model, dataset, use_adversarial=False):
    for x_batch, y_batch in dataset.take(1):
        if use_adversarial:
            x_batch = pgd_attack(model, x_batch, y_batch)
        loss, accuracy = model.evaluate(x_batch, y_batch, verbose=0)
        print(f"{'Adversarial' if use_adversarial else 'Clean'} accuracy: {accuracy * 100:.2f}%")
        return accuracy

# Run the demo
print("Creating and training MobileNetV2 with gradient obfuscation...")
mobilenet_model = create_mobilenetv2()
mobilenet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

train_with_gradient_obfuscation(mobilenet_model, train_ds, epochs=2)

print("\nEvaluating on clean test data...")
evaluate_model(mobilenet_model, test_ds, use_adversarial=False)

print("\nEvaluating on adversarial test data...")
evaluate_model(mobilenet_model, test_ds, use_adversarial=True)


Creating and training MobileNetV2 with gradient obfuscation...

Epoch 1/2
Step 0, Loss: 2.7183
Step 100, Loss: 2.1983
Step 200, Loss: 1.7305
Step 300, Loss: 1.5830
Step 400, Loss: 1.5260
Step 500, Loss: 1.5310
Step 600, Loss: 1.5843
Step 700, Loss: 2.3069
Step 800, Loss: 1.9900
Step 900, Loss: 1.6459
Step 1000, Loss: 1.5358
Step 1100, Loss: 1.4323
Step 1200, Loss: 1.4863
Step 1300, Loss: 1.3866
Step 1400, Loss: 1.4745
Step 1500, Loss: 1.1034

Epoch 2/2
Step 0, Loss: 1.2669
Step 100, Loss: 1.1174
Step 200, Loss: 1.1735
Step 300, Loss: 1.2627
Step 400, Loss: 1.3948
Step 500, Loss: 0.8522
Step 600, Loss: 1.2676
Step 700, Loss: 1.2355
Step 800, Loss: 1.0882
Step 900, Loss: 1.1730
Step 1000, Loss: 1.0042
Step 1100, Loss: 1.5131
Step 1200, Loss: 0.9583
Step 1300, Loss: 0.7883
Step 1400, Loss: 1.1571
Step 1500, Loss: 1.0800

Evaluating on clean test data...
Clean accuracy: 6.25%

Evaluating on adversarial test data...
Adversarial accuracy: 6.25%


0.0625